## Estimate the Planning Baseline from Historical Segment Data

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

REPORT_DIR = PROJECT_ROOT / "reports"

segment_comparison = pd.read_csv(
    REPORT_DIR / "retention_segment_comparison.csv"
)

planning_segment = segment_comparison.loc[
    segment_comparison["cohort"] == "later_evaluation"
].iloc[0]

baseline_rate = float(planning_segment["target_churn_rate"])
historical_candidates = int(planning_segment["target_users"])

ALPHA = 0.05
TARGET_POWER = 0.80
PRIMARY_MDE = 0.02

print(f"Planning baseline churn rate: {baseline_rate:.2%}")
print("Historical candidate users:", historical_candidates)
print(f"Primary absolute MDE: {PRIMARY_MDE:.2%}")

Planning baseline churn rate: 13.89%
Historical candidate users: 5041
Primary absolute MDE: 2.00%


## Power Analysis for Two Independent Proportions

Use a two-sided normal approximation with Cohen's h and equal allocation.
Effect sizes are planning assumptions, not measured treatment effects.

In [2]:
power_solver = NormalIndPower()

available_per_arm = historical_candidates // 2

power_rows = []

for absolute_reduction in [0.01, 0.02, 0.03]:
    treatment_rate = baseline_rate - absolute_reduction

    assert 0 < treatment_rate < baseline_rate < 1

    effect_size = abs(
        proportion_effectsize(baseline_rate, treatment_rate)
    )

    required_per_arm = int(np.ceil(
        power_solver.solve_power(
            effect_size=effect_size,
            alpha=ALPHA,
            power=TARGET_POWER,
            ratio=1.0,
            alternative="two-sided",
        )
    ))

    available_power = power_solver.power(
        effect_size=effect_size,
        nobs1=available_per_arm,
        alpha=ALPHA,
        ratio=1.0,
        alternative="two-sided",
    )

    power_rows.append({
        "absolute_reduction_pp": absolute_reduction * 100,
        "control_rate_assumption": baseline_rate,
        "treatment_rate_assumption": treatment_rate,
        "relative_reduction_pct": (
            100 * absolute_reduction / baseline_rate
        ),
        "required_per_arm": required_per_arm,
        "required_total": 2 * required_per_arm,
        "historical_candidate_users": historical_candidates,
        "power_with_balanced_historical_pool": available_power,
        "historical_pool_large_enough": (
            2 * required_per_arm <= historical_candidates
        ),
    })

power_analysis = pd.DataFrame(power_rows)
power_analysis

,absolute_reduction_pp,control_rate_assumption,treatment_rate_assumption,relative_reduction_pct,required_per_arm,required_total,historical_candidate_users,power_with_balanced_historical_pool,historical_pool_large_enough
0,1.0,0.138861,0.128861,7.201429,18195,36390,5041,0.180824,False
1,2.0,0.138861,0.118861,14.402857,4400,8800,5041,0.563744,False
2,3.0,0.138861,0.108861,21.604286,1887,3774,5041,0.899404,True


In [3]:
import json
import statsmodels

primary_plan = power_analysis.loc[
    np.isclose(
        power_analysis["absolute_reduction_pp"],
        PRIMARY_MDE * 100,
    )
].iloc[0]

required_per_arm = int(primary_plan["required_per_arm"])
required_total = int(primary_plan["required_total"])

recruitment_rows = []

for eligible_users_per_day in [50, 100, 200]:
    recruitment_days = int(np.ceil(
        required_total / eligible_users_per_day
    ))

    recruitment_rows.append({
        "hypothetical_eligible_unique_users_per_day": eligible_users_per_day,
        "recruitment_days": recruitment_days,
        "max_followup_after_last_enrollment_days": 60,
        "days_to_full_followup_excluding_ingestion_lag": (
            recruitment_days + 60
        ),
    })

recruitment_scenarios = pd.DataFrame(recruitment_rows)

power_analysis.to_csv(
    REPORT_DIR / "retention_power_analysis.csv",
    index=False,
)

recruitment_scenarios.to_csv(
    REPORT_DIR / "retention_recruitment_scenarios.csv",
    index=False,
)

experiment_config = {
    "status": "proposed; not run",
    "historical_baseline_rate": baseline_rate,
    "baseline_requires_prospective_calibration": True,
    "historical_candidate_users": historical_candidates,
    "absolute_mde": PRIMARY_MDE,
    "alpha": ALPHA,
    "target_power": TARGET_POWER,
    "alternative": "two-sided",
    "power_method": "normal approximation with Cohen's h",
    "allocation_ratio": 1.0,
    "required_per_arm": required_per_arm,
    "required_total": required_total,
    "randomization_unit": "user",
    "analysis_population": "intention-to-treat",
    "maximum_days_until_baseline_expiry": 30,
    "renewal_observation_days": 30,
    "maximum_followup_days": 60,
    "recruitment_scenarios_are_forecasts": False,
    "interim_efficacy_testing_planned": False,
    "statsmodels_version": statsmodels.__version__,
}

(REPORT_DIR / "retention_experiment_config.json").write_text(
    json.dumps(experiment_config, indent=2) + "\n",
    encoding="utf-8",
)

print("Power analysis and proposed experiment configuration saved.")
recruitment_scenarios

Power analysis and proposed experiment configuration saved.


,hypothetical_eligible_unique_users_per_day,recruitment_days,max_followup_after_last_enrollment_days,days_to_full_followup_excluding_ingestion_lag
0,50,176,60,236
1,100,88,60,148
2,200,44,60,104
